In [513]:
import numpy as np 
import random
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
from keras.utils import to_categorical

In [514]:
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]

In [515]:
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data


In [321]:
class Layer():
    def __init__(self, n_num, input_dim):
        self.weights = [0]*n_num
        self.weights=self.initialize_parameters(input_dim)
        

    def initialize_parameters(self, input_dim):
        """Initialize parameters with He initialization method"""
        
        parameters = {}
        parameters["W"] = np.random.randn(len(self.weights), input_dim) * np.sqrt(2/input_dim)
        parameters["b"] = np.zeros((len(self.weights), 1))

        return parameters



In [516]:
class FFNN():
    def __init__(self):
        self.layers=[]
        #self.weights = []
    
    def add_layer(self, n_num, input_dim=0, activation="relu"):
        if input_dim==0:
            input_dim=len(self.layers[-1][0].weights['W'])
        new_layer=Layer(n_num, input_dim)
        self.layers.append([new_layer,activation])
    def fit(self, X_train, Y_train, batch_size, epochs=0, learning_rate=0.001):

        self.minibatch_size = batch_size
        temp_weights=self.layers[1:].copy()
        for i in range(epochs):
            for batch_idx in range(0, int(X_train.shape[1] / batch_size)):
                # Mini Batch Samples
                if batch_idx == int(X_train.shape[1] / batch_size):
                    X = X_train[:, batch_idx*batch_size:]
                    Y = Y_train[:, batch_idx*batch_size:]
                else:
                    X = X_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]
                    Y = Y_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]

            I = X

            simple_weights=self.feed_forward(I, temp_weights)
            prediction= simple_weights["A"+str(len(temp_weights))]
        temp_weights.insert(0, self.layers[0])
        self.layers=temp_weights



In [323]:
model = FFNN()
model.add_layer(128, input_dim=784, activation='relu') # First hidden layer
model.add_layer(64, activation='relu')  # Second hidden layer
model.add_layer(32, activation='relu')  # Third hidden layer
model.add_layer(5, activation='softmax')  # Output layer for multi-class classification

#print(model.layers[0][0].weights['W0'].shape)
for l in model.layers:
    print(l[0].weights['W'], l[1])
    #print(l[0].weights['W'], l[1])
c=model.layers.copy()
print(model.layers)
print(c)

[[-0.05032772 -0.01651814  0.03359001 ...  0.01245094 -0.04191307
  -0.08151549]
 [-0.0362408   0.00280559 -0.05036998 ... -0.02402617 -0.01486406
   0.04699712]
 [ 0.02155376 -0.07233575  0.01476984 ...  0.01184866 -0.05701767
   0.05995346]
 ...
 [ 0.0433725  -0.02663502  0.00479488 ... -0.04682852 -0.00232392
  -0.1291247 ]
 [ 0.01892164 -0.01025813 -0.02564909 ...  0.05411753 -0.02790937
  -0.10414215]
 [ 0.00110153 -0.09879034 -0.00503972 ...  0.07839381 -0.10464476
   0.0442834 ]] relu
[[ 0.05503936  0.08806846 -0.03048719 ... -0.07598952 -0.01879231
   0.23935448]
 [-0.12955329  0.03068578  0.03894204 ...  0.11516581 -0.2034173
   0.01983434]
 [-0.05703942  0.25062296 -0.01273783 ... -0.06353586 -0.20395251
  -0.07556646]
 ...
 [ 0.12608957 -0.02605438 -0.18032513 ...  0.08867263  0.02174146
  -0.02914726]
 [-0.12038738 -0.32371123 -0.23560928 ...  0.08993868  0.18382914
   0.0576665 ]
 [ 0.09063764  0.25033638  0.06020949 ...  0.01732998  0.02591186
  -0.10025371]] relu
[[-0.02

In [324]:
l=[[1],[2],[3]]
temp_weights=l[1:].copy()
for i in model.layers:
    print(i[0].weights['W'].shape)
print(temp_weights)
metrics1={}
metrics=['accuracy', 'loss', 'recall']
for m in metrics: metrics1.update({m : []})
print(metrics1)
print(len(train_data_y.shape))

(128, 784)
(64, 128)
(32, 64)
(5, 32)
[[2], [3]]
{'accuracy': [], 'loss': [], 'recall': []}
1


In [442]:

for i in range(10,-1,-1):
    print(i)

10
9
8
7
6
5
4
3
2
1
0


In [ ]:
#define neural network layer
class Layer():
    def __init__(self, n_num, input_dim):
        self.weights = self.initialize_parameters(n_num, input_dim)
        

    def initialize_parameters(self, n_num, input_dim):
        """Initialize parameters with He initialization method"""
        
        parameters = {}
        parameters["W"] = np.random.randn(n_num, input_dim) * np.sqrt(2/input_dim)
        parameters["b"] = np.zeros((n_num, 1))

        return parameters
    
class Adam_optimizer():
    def __init__(self, learning_rate, beta1, beta2, epsilon, decay_rate):
        self.momentum_w = {}
        self.momentum_b = {}
        self.rms_w = {}
        self.rms_b = {}
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.learning_rate = learning_rate
        self.decay_rate = decay_rate  # Added decay rate
        self.t = 0  # Time step for bias correction


    def update_parameters(self, parameters, gradients, epoch):#(self, t, l_num, b, dw, db):
        ## Compute decayed learning rate
        
        ## dw, db are from current minibatch
        ## momentum beta 1
        # *** weights *** #
        self.t += 1
        decayed_learning_rate = self.learning_rate * np.exp(-self.decay_rate * epoch)
        
        # Update parameters for each layer
        for l_num, l in enumerate(parameters, start=1):
            #print(l_num)
            #print(gradients["dW" + str(l_num)].shape, )
            # Initialize caches if not already present
            if l_num not in self.momentum_w:
                self.momentum_w[l_num] = np.zeros_like(parameters[l_num-1][0].weights['W'])
                self.momentum_b[l_num] = np.zeros_like(parameters[l_num-1][0].weights['b'])
                self.rms_w[l_num] = np.zeros_like(parameters[l_num-1][0].weights['W'])
                self.rms_b[l_num] = np.zeros_like(parameters[l_num-1][0].weights['b'])
            # Momentum updates
            self.momentum_w[l_num] = self.beta1 * self.momentum_w[l_num] + (1 - self.beta1) * gradients["dW" + str(l_num)]
            self.momentum_b[l_num] = self.beta1 * self.momentum_b[l_num] + (1 - self.beta1) * gradients["db" + str(l_num)]
            
            # RMS updates
            self.rms_w[l_num] = self.beta2 * self.rms_w[l_num] + (1 - self.beta2) * (gradients["dW" + str(l_num)]**2)
            self.rms_b[l_num] = self.beta2 * self.rms_b[l_num] + (1 - self.beta2) * (gradients["db" + str(l_num)]**2)
            
            # Bias correction
            momentum_w_corr = self.momentum_w[l_num] / (1 - self.beta1**self.t)
            momentum_b_corr = self.momentum_b[l_num] / (1 - self.beta1**self.t)
            rms_w_corr = self.rms_w[l_num] / (1 - self.beta2**self.t)
            rms_b_corr = self.rms_b[l_num] / (1 - self.beta2**self.t)

            # Update weights and biases
            l[0].weights['W'] -= decayed_learning_rate * (momentum_w_corr / (np.sqrt(rms_w_corr) + self.epsilon))
            l[0].weights['b'] -= decayed_learning_rate * (momentum_b_corr / (np.sqrt(rms_b_corr) + self.epsilon))
            
        return parameters

class Mini_Batch_GD_optimizer():
    def __init__(self, learning_rate, decay_rate=0.01):
        #gradients=self.find_gradients(parameters, forward_vars, Y)
        #self.update_parameters(parameters, gradients, learning_rate)
        self.learning_rate=learning_rate
        self.decay_rate=decay_rate
    
    def update_parameters(self, parameters, gradients, epoch):
        decayed_learning_rate = self.learning_rate * np.exp(-self.decay_rate * epoch)
        l_num=0
        for l in parameters:
            l[0].weights['W'] -= decayed_learning_rate * gradients["dW"+str(l_num)]
            l[0].weights['b'] -= decayed_learning_rate * gradients["db"+str(l_num)]
            l_num+=1
        return parameters
    
#define fnn    
class FFNN():
    def __init__(self):
        self.layers=[]
        self.optimizer=None
        self.metrics={}
        #self.weights = []
    
    def add_layer(self, n_num, input_dim=None, activation="relu"):
        if input_dim is None:
            if len(self.layers) == 0:
                raise ValueError("Input dimension must be provided for the first layer.")
            input_dim = len(self.layers[-1][0].weights['W'])
        new_layer = Layer(n_num, input_dim)
        self.layers.append([new_layer, activation])
    def compile(self, optimizer='adam', learning_rate=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8, decay_rate=0.01, metrics=['accuracy']):
        if optimizer=='adam':
            self.optimizer=Adam_optimizer(learning_rate, beta1, beta2, epsilon, decay_rate)
        elif optimizer=='mini':
            self.optimizer=Mini_Batch_GD_optimizer(learning_rate, decay_rate)
        for m in metrics: self.metrics.update({m : []})
        
        
    def fit(self, X_train, Y_train, batch_size, epochs=0):

        self.minibatch_size = batch_size
        temp_weights=self.layers.copy()
        #for l in temp_weights:
            #print(l[0].weights["W"].shape)
        for i in range(epochs):
            for batch_idx in range(0, int(X_train.shape[0] / batch_size)):
                # Mini Batch Samples
                if batch_idx == int(X_train.shape[1] / batch_size):
                    X = X_train[:, batch_idx*batch_size:]
                    Y = Y_train[:, batch_idx*batch_size:]
                else:
                    X = X_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]
                    Y = Y_train[:, batch_idx*batch_size: (batch_idx+1)*batch_size]
                #print(X)
                output_history=self.feed_forward(temp_weights, X)
                prediction= output_history["A"+str(len(temp_weights))]
                #print(prediction.shape)
                self.get_stats(prediction, Y)
                
                #p=temp_weights[0][0].weights['W'][0].copy()
                #print("Weights before update:", temp_weights[-1][0].weights['W'].shape)
                #print("A before update:", output_history['A3'])
                gradients = self.backward_prop(temp_weights, output_history, Y)
                temp_weights = self.optimizer.update_parameters(temp_weights, gradients, epoch=i)
                #print("Weights after update:", temp_weights[0][0].weights['W'][0])
                #print("diff=", p- temp_weights[0][0].weights['W'][0])
        #temp_weights.insert(0, self.layers[0])
        self.layers=temp_weights
        

    
    def feed_forward(self, temp_weights, I):
        
        forward_vars = {"A0": I}
        for l_num in range(1,len(temp_weights)+1): 
            layer = temp_weights[l_num - 1] 
            #print("!", l_num, layer[1])   
                #I = np.dot(I, layer[0]) 
            forward_vars["Z"+str(l_num)] = np.dot(layer[0].weights['W'], forward_vars["A"+str(l_num-1)]) + layer[0].weights['b']
                
            if l_num == len(self.layers): 
                    forward_vars["A"+str(l_num)] = self._activate(forward_vars["Z"+str(l_num)], layer[1]) #output layer 
                    #print("forward_vars['A'+str(l_num)]", forward_vars["A"+str(l_num)].shape )
            else: 
                    forward_vars["A"+str(l_num)] = self._activate(forward_vars["Z"+str(l_num)], layer[1]) #hidden layers 
            l_num+=1        
        return forward_vars #{k: v for k, v in forward_vars.items() if k.startswith("A")}

    def backward_prop(self, parameters, forward_vars, Y):
        gradients = {}
        #L = len(self.layer_dims) - 1
        l_num=len(self.layers)
        for l_num in range(len(parameters), 0, -1):
            m = forward_vars["A"+str(l_num-1)].shape[1]
            #print(forward_vars["A"+str(l_num-1)])
            layer = parameters[l_num - 1]
            if layer[1] == 'sigmoid':
                gradients["dA" + str(l_num)] = -np.divide(Y, forward_vars["A" + str(l_num)]) + np.divide((1 - Y), (1 - forward_vars["A" + str(l_num)]))
                gradients["dZ" + str(l_num)] = gradients["dA" + str(l_num)] * forward_vars["A" + str(l_num)] * (1 - forward_vars["A" + str(l_num)])
            # other activations
            elif layer[1] == 'relu':
                relu_derivative = forward_vars["A"+str(l_num)] > 0
                gradients["dZ"+str(l_num)] = np.multiply(gradients["dA"+str(l_num)], relu_derivative)
            elif layer[1] == 'softmax':
                gradients["dZ"+str(l_num)] = forward_vars["A"+str(l_num)] - Y 

            gradients["dW" + str(l_num)] = (1 / m) * np.dot(gradients["dZ" + str(l_num)], forward_vars["A" + str(l_num - 1)].T)
            gradients["db" + str(l_num)] = (1 / m) * np.sum(gradients["dZ" + str(l_num)], axis=1, keepdims=True)
            gradients["dA" + str(l_num - 1)] = np.dot(layer[0].weights['W'].T, gradients["dZ" + str(l_num)])
        return gradients
    
    def predict(self, X):
            """Predict labels for given data X"""
            #print(self.layers[0][0].weights['W'])
            forward_vars = self.feed_forward(self.layers, X)
            
            L = len(self.layers)
            #print(forward_vars["A"+str(L)])
            return forward_vars["A"+str(L)]
    
    def _activate(self, I, activation):
        
        if activation=='relu':
            result=np.maximum(0, I)
        elif activation=='softmax':
            T = np.exp(I - np.max(I, axis=0, keepdims=True))
            T_sum = np.sum(T, axis=0, keepdims=True)
            result = np.divide(T, T_sum)
        elif activation=='sigmoid':
            result = 1 / (1 + np.exp(-I))
        else:
            print("Error")
        return result
    
    def get_stats(self, Y_hat, Y, num_classes=5):
        """
        Log Loss is applied
        """
    
        
        # Count correct predictions
        correct_predictions = np.sum(Y == Y_hat)
        total_predictions = len(Y)
        
        # Calculate accuracy
       
        TP = np.zeros(num_classes, dtype=int)
        FN = np.zeros(num_classes, dtype=int)
        
        # Calculate TP and FN for each class
        for i in range(num_classes):
            TP[i] = np.sum((Y == i) & (Y_hat == i))  # True Positives for class i
            FN[i] = np.sum((Y == i) & (Y_hat != i))  # False Negatives for class i
        
        # Calculate recall for each class
        recall_per_class = {}
        for i in range(num_classes):
            if TP[i] + FN[i] == 0:
                recall_per_class[i] = 0.0  # Avoid division by zero
            else:
                recall_per_class[i] = TP[i] / (TP[i] + FN[i])
        
        m = Y.shape[1]
        self.metrics['loss'].append((1/m) * np.sum(-(Y * np.log(Y_hat + 1e-15))))
        self.metrics['accuracy'].append(correct_predictions / total_predictions)
        self.metrics['recall'].append(recall_per_class)




        
        
         
    
    

In [393]:
a = to_categorical([0, 1, 2, 3, 3], num_classes=5)
def one_hot_encode(Y, num_classes):
    return np.eye(num_classes)[Y]
b=one_hot_encode([0, 1, 2, 3, 3], 5)

print(a)
print(np.argmax(a, axis=1))
print(b)
print(np.argmax(b, axis=1))

[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]]
[0 1 2 3 3]
[[1. 0. 0. 0. 0.]
 [0. 1. 0. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 [0. 0. 0. 1. 0.]]
[0 1 2 3 3]


In [556]:
def create_multiclass_model(input_dim, num_classes):
    model = FFNN()
    model.add_layer(256, input_dim=input_dim, activation='relu') # First hidden layer
    model.add_layer(128, activation='relu') # First hidden layer
    model.add_layer(64, activation='relu')  # Second hidden layer
    model.add_layer(32, activation='relu')  # Third hidden layer
    model.add_layer(num_classes, activation='softmax')  # Output layer for multi-class classification
  
    # Compile the model
    model.compile(optimizer='adam',learning_rate=0.001,
                 #loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
                 metrics=['accuracy', 'loss', 'recall'])
    return model

In [557]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []
def one_hot_encode(Y, num_classes):
    return np.eye(num_classes)[Y]

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index].T, train_data_X[val_index].T
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    y_train = one_hot_encode(y_train, 5).T
   
    #print(y_train)
    #y_train=y_train.reshape(1,len(y_train))
    #y_val=y_val.reshape(1,len(y_val))
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=len(np.unique(train_data_y)))
    #print(X_val.shape, X_train.shape)
    # Train the model
    model.fit(X_train, y_train, epochs=100, batch_size=32)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=0)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    print(y_val_pred.shape)
    
    #print(y_val.shape)
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1
(2000,)
Fold 1 Accuracy: 0.8070
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.80      0.79      0.79       415
           1       0.96      0.96      0.96       379
           2       0.77      0.84      0.80       388
           3       0.89      0.87      0.88       405
           4       0.63      0.60      0.61       413

    accuracy                           0.81      2000
   macro avg       0.81      0.81      0.81      2000
weighted avg       0.81      0.81      0.81      2000

Confusion Matrix for Fold 1:
 [[329   2  14   8  62]
 [  4 362   1   9   3]
 [  1   0 326   9  52]
 [ 10   9   9 351  26]
 [ 69   5  75  18 246]]
Training fold 2
(2000,)
Fold 2 Accuracy: 0.7945
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.72      0.78      0.75       398
           1       0.97      0.95      0.96       393
           2       0.84      0.81   

In [550]:
def create_multiclass_model(input_dim, num_classes):
    model = FFNN()
    model.add_layer(256, input_dim=input_dim, activation='relu') # First hidden layer
    model.add_layer(128, activation='relu') # First hidden layer
    model.add_layer(64, activation='relu')  # Second hidden layer
    model.add_layer(32, activation='relu')  # Third hidden layer
    model.add_layer(num_classes, activation='softmax')  # Output layer for multi-class classification
  
    # Compile the model
    model.compile(optimizer='adam',learning_rate=0.001,
                 #loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
                 metrics=['accuracy', 'loss', 'recall'])
    return model

In [551]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []
def one_hot_encode(Y, num_classes):
    return np.eye(num_classes)[Y]
    # Split the data into training and validation sets for the current fold
    
for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index].T, X_new_reduced[val_index].T
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    y_train = one_hot_encode(y_train, 5).T
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=X_new_reduced.shape[1], num_classes=len(np.unique(train_data_y)))
    #print(X_val.shape, X_train.shape)
    
    # Train the model
    model.fit(X_train, y_train, epochs=100, batch_size=32)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=0)  # Get the predicted class for each sample
    #print(y_val_pred.shape)
    #print(y_val.shape)
    # Calculate accuracy for the current fold
    accuracy_pca = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy_pca)
    
    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy_pca = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1
Fold 1 Accuracy: 0.8015
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.37      0.65      0.47       415
           1       0.50      0.50      0.50       379
           2       0.40      0.40      0.40       388
           3       0.64      0.46      0.54       405
           4       0.33      0.16      0.22       413

    accuracy                           0.44      2000
   macro avg       0.45      0.44      0.43      2000
weighted avg       0.45      0.44      0.42      2000

Confusion Matrix for Fold 1:
 [[270  11  34  26  74]
 [130 190  32  21   6]
 [ 93  90 157  16  32]
 [100  44  45 187  29]
 [132  47 122  44  68]]
Training fold 2
Fold 2 Accuracy: 0.8015
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.75      0.52      0.61       398
           1       0.65      0.92      0.76       393
           2       0.53      0.80      0.63       42

In [552]:
print(model.metrics)

{'accuracy': [0.6, 1.8, 3.8, 4.4, 4.6, 4.8, 5.4, 5.8, 5.8, 5.8, 6.2, 6.6, 6.2, 6.6, 6.6, 6.6, 6.6, 6.4, 6.6, 6.6, 6.6, 6.6, 6.8, 6.8, 6.8, 6.8, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.0, 7.2, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4, 7.4], 'loss': [30.22213034879243, 17.667206979918493, 9.018896499887692, 3.2069226736585548, 3.8396067870532873, 3.2380101788279276, 1.0793367634421356, 1.0793368321147874, 0.26160074051457965, 0.11448180144501376, 0.16357669407123826, 1.6552060716838736e-05, 1.9817458982904295e-07, 6.4086995454621e-09, 0.03117266653263795, 0.004660140836723422, 1.536711931188226e-05, 8.506097267764738e-08, 1.812206751564739e-09, 1.0143455695142145e-08, 8.06425090716435e-08, 5.513015890883587e-07, 3.808